### Installing required packages    

In [15]:
!pip install google-cloud-storage
!pip install --upgrade google-cloud-storage


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


### Google cloud 

In [16]:
from google.cloud import storage

In [17]:
try:
    # If you need to specify a project explicitly:
    # storage_client = storage.Client(project=PROJECT_ID)
    storage_client = storage.Client()
    print("Google Cloud Storage client initialized successfully.")
except Exception as e:
    print(f"Error initializing GCS client: {e}")
    print(
        "Please ensure your authentication is set up correctly (e.g., `gcloud auth application-default login` or Colab authentication)."
    )
    exit()  # Exit if client cannot be initialized

# --- 1. List all buckets in your project ---
print("\n--- Listing all buckets in your project ---")
try:
    buckets = storage_client.list_buckets()
    for bucket in buckets:
        print(f"Bucket Name: {bucket.name}")
except Exception as e:
    print(f"Error listing buckets: {e}")

Google Cloud Storage client initialized successfully.

--- Listing all buckets in your project ---
Bucket Name: apps-cityvision-prod_cloudbuild
Bucket Name: archive-cityvision
Bucket Name: dataflow-cityvision
Bucket Name: gcf-v2-sources-220669230854-us-west1
Bucket Name: open-cityvision
Bucket Name: processed-cityvision
Bucket Name: raw-cityvision
Bucket Name: run-sources-apps-cityvision-prod-us-west1


In [18]:
storage_client = storage.Client()
bucket = storage_client.bucket("processed-cityvision")
blobs = bucket.list_blobs()
print(f"the blobs in the bucket {bucket} are :")
for blob in blobs:
    print(blob.name)
    break

the blobs in the bucket <Bucket: processed-cityvision> are :
142edc03-362c-57e1-8400-858886f38a8c/annotated_videos/SCU2VN_202410012345_000.mp4


### YOLO Training Code

#### Pip Install reqs

In [19]:
!pip install google-cloud-storage
!pip install ultralytics


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
!pip install ultralytics


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Configuring GCLOUD for the Jupyter Notebook

In [21]:
import os

print(os.environ["PATH"])

# run this command on the terminal and compare the output with the notebook - "which gcloud"

C:\Users\MJATTU\Desktop\Projects\cityvision\venv\Lib\site-packages\cv2\../../x64/vc17/bin;c:\Users\MJATTU\Desktop\Projects\cityvision\venv\Scripts;C:\Users\MJATTU\Desktop\Projects\cityvision\venv\Scripts;C:\Users\MJATTU\AppData\Local\cloud-code\installer\google-cloud-sdk\bin;C:\Program Files\Alacritty\;C:\WINDOWS\system32;C:\WINDOWS;C:\WINDOWS\System32\Wbem;C:\WINDOWS\System32\WindowsPowerShell\v1.0\;C:\WINDOWS\System32\OpenSSH\;C:\Program Files (x86)\Pulse Secure\VC142.CRT\X64\;C:\Program Files (x86)\Pulse Secure\VC142.CRT\X86\;C:\Program Files (x86)\Common Files\Pulse Secure\TNC Client Plugin\;\\CEPFILE2\apps\ORACLE\Ora1210x86\BIN;C:\Program Files\dotnet\;C:\Program Files\Git\cmd;C:\Users\MJATTU\AppData\Local\Programs\Python\Python313\python.exe;;C:\ProgramData\chocolatey\bin;C:\Program Files\Docker\Docker\resources\bin;C:\Users\MJATTU\AppData\Local\Programs\Python\Python313\Scripts\;C:\Users\MJATTU\AppData\Local\Programs\Python\Python313\;C:\Users\MJATTU\AppData\Local\Programs\Pytho

In [22]:
# adding the gcloud command to the PATH
import os

os.environ["PATH"] += (
    os.pathsep + "/home2/mikjat/projects/newdir/cityvision/google-cloud-sdk/bin"
)

In [23]:
# checking if the gcloud command works
!gcloud --version

Google Cloud SDK 530.0.0
bq 2.1.19
core 2025.07.11
gcloud-crc32c 1.0.0
gsutil 5.35


Updates are available for some Google Cloud CLI components.  To install them,
please run:
  $ gcloud components update


In [24]:
# Cell 1: Set the environment variable and then authenticate

import os
import google.auth

# Use os.path.expanduser() to correctly expand the "~" character
credentials_path = os.path.expanduser(
    "~/.config/gcloud/application_default_credentials.json"
)

# Check if the file exists before setting the environment variable
if os.path.exists(credentials_path):
    os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = credentials_path
    print(
        f"GOOGLE_APPLICATION_CREDENTIALS environment variable set to: {credentials_path}"
    )

    # Now, attempt to authenticate
    try:
        credentials, project = google.auth.default()
        print("Authenticated with Google Cloud successfully.")
        print(f"Using credentials for project: {project}")
    except Exception as e:
        print(f"Authentication failed: {e}")
        # This will give a more descriptive error if authentication fails for other reasons.
else:
    print(f"Error: Credentials file not found at {credentials_path}")
    print(
        "Please ensure you have run 'gcloud auth application-default login' in your terminal."
    )

Error: Credentials file not found at C:\Users\MJATTU/.config/gcloud/application_default_credentials.json
Please ensure you have run 'gcloud auth application-default login' in your terminal.


#### Training Code

In [26]:
# Import necessary libraries
import os
import subprocess
from google.cloud import storage
from ultralytics import YOLO
import yaml
import shutil
import random
import cv2
import numpy as np
import zipfile
import yaml
from typing import Union, Tuple
import sys
from ultralytics import settings

In [27]:
# --- Configuration ---
GCS_BUCKET_NAME = "open-cityvision"
GCS_DATA_PATH = "."  # Path *within* your GCS bucket to the dataset root
LOCAL_DATA_DIR = os.getcwd()  # the current directory where the data will be downloaded

In [28]:
# YOLO Model Configuration
IMG_SIZE = 640
BATCH_SIZE = 64
DEVICE = 0
EPOCHS = 50

In [29]:
def download_data_from_gcs(bucket_name: str, gcs_path: str, local_dir: str) -> None:
    """
    Downloads a directory and its contents from a Google Cloud Storage bucket.

    Args:
        bucket_name (str): The name of the GCS bucket.
        gcs_path (str): The path to the directory in GCS (e.g., "datasets/my_data/").
        local_dir (str): The local directory to save the downloaded files.
    """
    # if the gcs_path is just ".", we want to list all blobs in the bucket
    if gcs_path == "" or gcs_path == ".":
        prefix = ""
    else:
        # Ensure the prefix ends with a '/' to treat it as a directory
        prefix = gcs_path.rstrip("/") + "/"

    print(
        f"Attempting to download data from gs://{bucket_name}/{gcs_path} to {local_dir}"
    )

    try:
        storage_client = storage.Client()
        bucket = storage_client.bucket(bucket_name)

        # Ensure local directory exists
        os.makedirs(local_dir, exist_ok=True)

        blobs = bucket.list_blobs(
            prefix=prefix, delimiter="/"
        )  # List all blobs with the given prefix
        downloaded_count = 0
        for blob in blobs:
            # Check if the blob is a .zip file and not a directory itself
            if blob.name.endswith(".zip"):
                # Construct local file path, taking only the base file name
                local_file_name = os.path.basename(blob.name)
                local_file_path = os.path.join(local_dir, local_file_name)

                print(f"Downloading {blob.name} to {local_file_path}")
                blob.download_to_filename(local_file_path)
                downloaded_count += 1
        if downloaded_count == 0:
            print(
                f"No zipped files found or downloaded from gs://{bucket_name}/{gcs_path}. "
                f"Please check bucket name and GCS path, and ensure there are .zip files present."
            )
        else:
            print(f"Successfully downloaded {downloaded_count} zipped files from GCS.")

    except Exception as e:
        print(f"An error occurred: {e}")
        print(f"Error downloading data from GCS: {e}")
        print("Please ensure your Google Cloud credentials are set up correctly.")
        print(
            "You can use `gcloud auth application-default login` for local development or set the "
            "`GOOGLE_APPLICATION_CREDENTIALS` environment variable for service accounts."
        )
        exit(1)  # Exit if data download fails

In [30]:
# unzipping files
def unzipDataset(folderPath: str) -> None:
    """
    Unzips all .zip files in the specified folder.
    Args:
        folderPath (str): The path to the folder containing .zip files.
    """
    files = os.listdir(folderPath)
    for file in files:
        if not file.endswith(".zip"):
            continue
        filename = file.split(".")[0]
        fullPath = os.path.join(folderPath, file)
        if os.path.isfile(fullPath):
            with zipfile.ZipFile(fullPath, "r") as zip_ref:
                zip_ref.extractall(os.path.join(folderPath, filename))

        os.remove(fullPath)
    return filename

In [31]:
# --- 2. Function to Train YOLO Model ---
def train_yolo_model(
    data_yaml_path: str,
    model: YOLO,
    epochs: int,
    img_size: int,
    batch_size: int,
    device: str,
    wandb: bool = False,
    hsv_h_range: Union[float, Tuple[float, float]] = 0.05,
    hsv_s_range: Union[float, Tuple[float, float]] = 0.5,
    hsv_v_range: Union[float, Tuple[float, float]] = 0.3,
) -> None:
    """
    Trains a YOLO model with specified parameters.
    Args:
        data_yaml_path (str): Path to the data.yaml file.
        model (YOLOModel): YOLO model instance to train.
        epochs (int): Number of training epochs.
        img_size (int): Image size for training.
        batch_size (int): Batch size for training.
        device (str): Device to train on (e.g., '0' for GPU, 'cpu').
        hsv_h_range (Union[float, Tuple[float, float]]): Hue augmentation range.
        hsv_s_range (Union[float, Tuple[float, float]]): Saturation augmentation range.
        hsv_v_range (Union[float, Tuple[float, float]]): Brightness augmentation range.
    """
    try:
        print(f"\n--- Starting YOLO Model Training with {model} ---")
        if wandb:
            print("WandB logging is enabled.")
        else:
            print("WandB logging is disabled. Training will not log to WandB.")
        # Train the model
        results = model.train(
            data=data_yaml_path,
            epochs=epochs,
            imgsz=img_size,
            batch=batch_size,
            device=device,
            val=False,  # Enable validation during training
            # --- ONLY HSV Augmentations ---
            hsv_h=hsv_h_range,  # Hue augmentation (randomly adjusted within +/- hsv_h_range)
            hsv_s=hsv_s_range,  # Saturation augmentation (randomly adjusted within +/- hsv_s_range)
            hsv_v=hsv_v_range,  # Brightness augmentation (randomly adjusted within +/- hsv_v_range)
            # --- Disable Other Augmentations ---
            degrees=0.0,  # Image rotation
            translate=0.0,  # Image translation
            scale=0.0,  # Image scaling
            shear=0.0,  # Image shearing
            perspective=0.0,  # Image perspective transformation
            flipud=0.0,  # Flip image upside down
            fliplr=1.0,  # Flip image left-right
            mosaic=0.0,  # Disable mosaic augmentation
            mixup=0.0,  # Disable mixup augmentation
            copy_paste=0.0,  # Disable copy-paste augmentation
            project="ultralytics_yolo_project",
            name="yolov11m_run",
            # auto_augment=None # Ensure auto_augment is not overriding
        )

        print("\n--- Training Complete! ---")
        print(f"Results saved to: {model.trainer.save_dir}")
        print("You can find the best.pt and last.pt models in this directory.")

    except Exception as e:
        print(f"Error during YOLO model training: {e}")
        print(
            "Please ensure Ultralytics is installed (`pip install ultralytics`) "
            "and your dataset `data.yaml` is correctly configured."
        )
        exit(1)

In [32]:
def train_with_data_in_cloud():
    # 1. Download data from GCS
    download_data_from_gcs(GCS_BUCKET_NAME, GCS_DATA_PATH, LOCAL_DATA_DIR)
    # 2. Unzip the downloaded files
    dataset_name = unzipDataset(LOCAL_DATA_DIR)
    # get the location of the data.yaml file
    yaml_path = os.path.join(dataset_name, "data.yaml")
    # 3. Train the YOLO model
    model = YOLO("yolo11m.pt")
    train_yolo_model(
        data_yaml_path=yaml_path,
        model=model,
        epochs=EPOCHS,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        device=DEVICE,
        wandb=True,  # Enable WandB logging
    )
    return model
def train_with_data_locally(dataset_location):
    # get the location of the data.yaml file
    yaml_path = os.path.join(dataset_location, "data.yaml")
    # 3. Train the YOLO model
    model = YOLO("yolo11l.pt")
    train_yolo_model(
        data_yaml_path=yaml_path,
        model=model,
        epochs=EPOCHS,
        img_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        device=DEVICE,
    )
    return model


In [33]:
# --- Main Execution Flow ---
if __name__ == "__main__":

    # train model based on where the data is
    # model = train_with_data_in_cloud()
    model = train_with_data_locally(os.path.join("yolo_dataset", "2025-1-07_len_14200"))

"""
Dataset '2025-08-11_len_3833/data.yaml' images not found ⚠️, missing path '/home2/mikjat/projects/newdir/datasets/2025-08-11_len_3833/val.txt'
Note dataset download directory is '/home2/mikjat/projects/newdir/datasets'. You can update this in '/home2/mikjat/.config/Ultralytics/settings.json'
Please ensure Ultralytics is installed (`pip install ultralytics`) and your dataset `data.yaml` is correctly configured.

if this error comes up then you will have to chnge the path of the dataset in the settings.json file

"""


--- Starting YOLO Model Training with YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(64, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C3k2(
        (cv1): Conv(
          (conv): Conv2d(128, 128, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(128, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(256, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(256, eps=0.

"\nDataset '2025-08-11_len_3833/data.yaml' images not found ⚠️, missing path '/home2/mikjat/projects/newdir/datasets/2025-08-11_len_3833/val.txt'\nNote dataset download directory is '/home2/mikjat/projects/newdir/datasets'. You can update this in '/home2/mikjat/.config/Ultralytics/settings.json'\nPlease ensure Ultralytics is installed (`pip install ultralytics`) and your dataset `data.yaml` is correctly configured.\n\nif this error comes up then you will have to chnge the path of the dataset in the settings.json file\n\n"

#### Side testing

In [ ]:
print("data yaml path is: ", yaml_path)
train_yolo_model(
    data_yaml_path=yaml_path,
    model=model,
    epochs=EPOCHS,
    img_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    device=DEVICE,
    wandb=False,
)

NameError: name 'yaml_path' is not defined

In [ ]:
# check if the gpu is available
import torch

if torch.cuda.is_available():
    DEVICE = 0
else:
    DEVICE = "cpu"

print("Using device:", DEVICE)

Using device: 0


In [ ]:
# checking the image size
import cv2


def check_image_size(image_path: str) -> Tuple[int, int]:
    """
    Checks the dimensions of an image.

    Args:
        image_path (str): Path to the image file.

    Returns:
        Tuple[int, int]: Width and height of the image.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Could not read image at {image_path}")
    return img.shape[1], img.shape[0]  # (width, height)


print(
    check_image_size(
        "2025-08-11_len_3833/train/images/SCU3I3_202405091745_018_image_9.jpg"
    )
)

In [ ]:
import os

print(f"Current working directory from Python: {os.getcwd()}")
print(f"Does 'yolov11m.pt' exist in current directory? {os.path.exists('yolo11m.pt')}")

Current working directory from Python: c:\Users\MJATTU\Desktop\Projects\cityvision\notebooks
Does 'yolov11m.pt' exist in current directory? True


## testing

In [ ]:
!pip install pytest pytest-mock google-cloud-storage


   ---------------------------------------- 0/4 [pluggy]
   ---------------------------------------- 0/4 [pluggy]
   ---------- ----------------------------- 1/4 [iniconfig]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ------------------- 2/4 [pytest]
   -------------------- ---

In [ ]:
import pytest
import os
import shutil
import sys

# --- Pytest Fixtures leveraging 'mocker' ---


@pytest.fixture
def mock_gcs_client(mocker):
    """
    Fixture to mock google.cloud.storage.Client using pytest-mock's mocker.
    The 'mocker' fixture is provided by pytest-mock.
    """
    # mocker.patch() is equivalent to unittest.mock.patch() but with pytest's
    # automatic teardown for mocks.
    mock_client = mocker.patch("google.cloud.storage.Client")
    mock_bucket = (
        mocker.Mock()
    )  # mocker.Mock() is also available, or plain unittest.mock.Mock()
    mock_client.return_value.bucket.return_value = mock_bucket
    return mock_client, mock_bucket


@pytest.fixture
def mock_sys_exit(mocker):
    """
    Fixture to mock sys.exit using pytest-mock's mocker.
    """
    return mocker.patch("sys.exit")


@pytest.fixture
def capsys_output(capsys):
    """
    Fixture to capture stdout/stderr. 'capsys' is a built-in pytest fixture,
    not specifically from pytest-mock, but commonly used alongside mocking.
    """
    return capsys


# --- Pytest Test Functions ---


def test_successful_download_of_zip_files(mock_gcs_client, tmp_path, capsys_output):
    """Test that only .zip files are downloaded successfully."""
    mock_client, mock_bucket = mock_gcs_client
    mock_bucket_name = "test-bucket"
    mock_gcs_path = "data/zipped_files/"
    local_dir = tmp_path / "test_downloads"  # tmp_path provides a unique temp dir

    # Create mock blob objects. mocker.Mock() can be used here too.
    mock_blob_zip1 = (
        mock_client.Mock()
    )  # Using mock_client.Mock() is also an option if you like
    mock_blob_zip1.name = "data/zipped_files/file1.zip"
    mock_blob_zip1.download_to_filename = mock_client.Mock()

    mock_blob_zip2 = mock_client.Mock()
    mock_blob_zip2.name = "data/zipped_files/subfolder/file2.zip"
    mock_blob_zip2.download_to_filename = mock_client.Mock()

    mock_blob_txt = mock_client.Mock()
    mock_blob_txt.name = "data/zipped_files/readme.txt"
    mock_blob_txt.download_to_filename = mock_client.Mock()

    mock_blob_dir = mock_client.Mock()
    mock_blob_dir.name = "data/zipped_files/subfolder/"
    mock_blob_dir.download_to_filename = mock_client.Mock()

    mock_bucket.list_blobs.return_value = [
        mock_blob_zip1,
        mock_blob_zip2,
        mock_blob_txt,
        mock_blob_dir,
    ]

    download_data_from_gcs(
        mock_bucket_name, mock_gcs_path, str(local_dir)
    )  # Pass string path

    # Assertions using standard 'assert' and mock methods
    mock_client.assert_called_once()
    mock_client.return_value.bucket.assert_called_once_with(mock_bucket_name)
    mock_bucket.list_blobs.assert_called_once_with(prefix=mock_gcs_path)

    mock_blob_zip1.download_to_filename.assert_called_once_with(
        os.path.join(str(local_dir), "file1.zip")
    )
    mock_blob_zip2.download_to_filename.assert_called_once_with(
        os.path.join(str(local_dir), "file2.zip")
    )
    assert not mock_blob_txt.download_to_filename.called  # Using .called for not called
    assert not mock_blob_dir.download_to_filename.called

    assert local_dir.is_dir()  # pytest's tmp_path returns a Path object

    captured = capsys_output.readouterr()
    # You can assert on specific print messages if the function prints success messages
    # assert "Successfully downloaded 2 zipped files from GCS." in captured.out


def test_no_zip_files_found(mock_gcs_client, tmp_path, capsys_output):
    """Test the scenario where no .zip files are found."""
    mock_client, mock_bucket = mock_gcs_client
    mock_bucket_name = "test-bucket"
    mock_gcs_path = "data/empty_dir/"
    local_dir = tmp_path / "test_downloads"

    mock_bucket.list_blobs.return_value = []

    download_data_from_gcs(mock_bucket_name, mock_gcs_path, str(local_dir))

    captured = capsys_output.readouterr()
    assert (
        f"No zipped files found or downloaded from gs://{mock_bucket_name}/{mock_gcs_path}."
        in captured.out
    )
    assert local_dir.is_dir()


def test_gcs_exception_handling(
    mock_gcs_client, mock_sys_exit, tmp_path, capsys_output
):
    """Test that exceptions from GCS operations are caught and handled."""
    mock_client, _ = mock_gcs_client
    mock_bucket_name = "test-bucket"
    mock_gcs_path = "data/error_path/"
    local_dir = tmp_path / "test_downloads"

    # Set the side_effect on the mocked Client to raise an exception
    mock_client.side_effect = Exception("Mock GCS connection error")

    download_data_from_gcs(mock_bucket_name, mock_gcs_path, str(local_dir))

    captured = capsys_output.readouterr()
    assert "An error occurred: Mock GCS connection error" in captured.out
    assert "Error downloading data from GCS: Mock GCS connection error" in captured.out
    assert (
        "Please ensure your Google Cloud credentials are set up correctly."
        in captured.out
    )

    # Assert that sys.exit(1) was called via the mocked sys.exit
    mock_sys_exit.assert_called_once_with(1)


def test_gcs_list_blobs_exception(
    mock_gcs_client, mock_sys_exit, tmp_path, capsys_output
):
    """Test exception during list_blobs call."""
    _, mock_bucket = mock_gcs_client  # We only need mock_bucket here
    mock_bucket_name = "test-bucket"
    mock_gcs_path = "data/error_listing/"
    local_dir = tmp_path / "test_downloads"

    # Set the side_effect on the mocked list_blobs method
    mock_bucket.list_blobs.side_effect = Exception("Mock list_blobs error")

    download_data_from_gcs(mock_bucket_name, mock_gcs_path, str(local_dir))

    captured = capsys_output.readouterr()
    assert "An error occurred: Mock list_blobs error" in captured.out
    assert "Error downloading data from GCS: Mock list_blobs error" in captured.out

    mock_sys_exit.assert_called_once_with(1)


def test_gcs_download_to_filename_exception(
    mock_gcs_client, mock_sys_exit, tmp_path, capsys_output
):
    """Test exception during download_to_filename call."""
    _, mock_bucket = mock_gcs_client  # We only need mock_bucket here
    mock_bucket_name = "test-bucket"
    mock_gcs_path = "data/error_downloading/"
    local_dir = tmp_path / "test_downloads"

    # Create a mock blob and set its download_to_filename side_effect
    mock_blob_zip = mock_gcs_client[
        0
    ].Mock()  # Access the MockerFixture's Mock() method
    mock_blob_zip.name = "data/error_downloading/corrupt.zip"
    mock_blob_zip.download_to_filename.side_effect = Exception("Mock download error")

    mock_bucket.list_blobs.return_value = [mock_blob_zip]

    download_data_from_gcs(mock_bucket_name, mock_gcs_path, str(local_dir))

    captured = capsys_output.readouterr()
    assert "An error occurred: Mock download error" in captured.out
    assert "Error downloading data from GCS: Mock download error" in captured.out

    mock_sys_exit.assert_called_once_with(1)